In [ ]:
!pip install -q scikit-learn


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# ==========================
# LOAD DATASET
# ==========================
df = pd.read_csv("/kaggle/input/datasets/madhusudhanpallela/preprocessed/preprocessed_dataset.csv", low_memory=True)

TARGET = "diseases"

# ==========================
# REMOVE NULL TARGETS
# ==========================
df = df.dropna(subset=[TARGET])

# ==========================
# REDUCE MEMORY
# ==========================
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = df[col].astype('float32')

for col in df.select_dtypes(include=['int64']).columns:
    df[col] = df[col].astype('int32')

# ==========================
# FEATURES & TARGET
# ==========================
X = df.drop(columns=[TARGET])
y = df[TARGET]

# ==========================
# REMOVE CLASSES HAVING <2 SAMPLES
# ==========================
counts = y.value_counts()
valid_classes = counts[counts >= 3].index

X = X[y.isin(valid_classes)]
y = y[y.isin(valid_classes)]

# ==========================
# COLUMN TYPES
# ==========================
numeric_features = X.select_dtypes(include=["int32","float32","int64","float64"]).columns

categorical_features = X.select_dtypes(include=["object","category"]).columns

# ==========================
# PREPROCESSING
# ==========================
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# ==========================
# SPLIT
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================
# BASE MODELS
# ==========================
estimators = [

    ("rf",
     RandomForestClassifier(
         n_estimators=50,
         max_depth=10,
         random_state=42,
         n_jobs=-1
     )),

    ("lr",
     LogisticRegression(
         max_iter=500,
         random_state=42
     ))
]

# ==========================
# STACKING MODEL
# ==========================
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=500),
    cv=3,
    n_jobs=-1
)

# ==========================
# COMPLETE PIPELINE
# ==========================
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", stack)
])

# ==========================
# TRAIN
# ==========================
print("Training Started...")

model.fit(X_train, y_train)

print("Training Completed")

# ==========================
# PREDICT
# ==========================
y_pred = model.predict(X_test)

print("\nAccuracy :", accuracy_score(y_test, y_pred))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred))

In [ ]:
print(pd.Series(y).value_counts())